# Dataset and dependencies

In [ ]:
from pathlib import Path
import csv

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

working_dir = Path.cwd()
dataset_candidates = [
    working_dir / "dataset videos",
    working_dir.parent / "dataset videos",
    working_dir.parent.parent / "dataset videos",
]
dataset_videos = next(
    (path for path in dataset_candidates if path.is_dir()),
    None,
)
if dataset_videos is None:
    raise FileNotFoundError(
        "Could not find the 'dataset videos' folder. "
        "Expected it beside the Badminton AI project folder."
    )

video_paths = sorted(dataset_videos.rglob("*.mp4"))
video_csv_pairs = [
    (video_path, video_path.with_suffix(".csv"))
    for video_path in video_paths
    if video_path.with_suffix(".csv").is_file()
]

print(f"Dataset folder: {dataset_videos}")
print(f"Videos found: {len(video_paths)}")
print(f"Videos with matching CSV annotations: {len(video_csv_pairs)}")

# Load annotations for a selected video as rows keyed by CSV column name.
def load_annotations(video_path):
    csv_path = video_path.with_suffix(".csv")
    with csv_path.open(newline="", encoding="utf-8") as annotation_file:
        return list(csv.DictReader(annotation_file))

if video_csv_pairs:
    sample_video, sample_csv = video_csv_pairs[0]
    sample_annotations = load_annotations(sample_video)
    print(f"Sample video: {sample_video.name}")
    print(f"Sample annotations: {len(sample_annotations)} frames from {sample_csv.name}")


## Prepare and train the shuttlecock detector

The CSV files contain shuttle center points, not bounding boxes. The preparation cell below converts those points into small YOLO boxes and splits by video to keep validation clips separate from training clips. The training step can take a while, especially on CPU.

In [ ]:
import random
import yaml

project_root = next(
    (path for path in [working_dir, *working_dir.parents] if (path / "yolov8x.pt").is_file()),
    working_dir.parent,
)
dataset_output = project_root / "training" / "shuttlecock_yolo_dataset"
FRAME_STRIDE = 4
SHUTTLE_BOX_SIZE = 20


def write_yolo_split(pairs, split_name):
    image_dir = dataset_output / "images" / split_name
    label_dir = dataset_output / "labels" / split_name
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    saved_count = 0

    for video_path, csv_path in pairs:
        with csv_path.open(newline="", encoding="utf-8") as annotation_file:
            annotations = {
                int(row["Frame"]): row
                for row in csv.DictReader(annotation_file)
            }
        if not annotations:
            continue

        capture = cv2.VideoCapture(str(video_path))
        if not capture.isOpened():
            print(f"Skipping unreadable video: {video_path}")
            continue

        frame_index = 0
        last_annotated_frame = max(annotations)
        while frame_index <= last_annotated_frame:
            success, frame = capture.read()
            if not success:
                break

            row = annotations.get(frame_index)
            if row is not None and frame_index % FRAME_STRIDE == 0:
                image_height, image_width = frame.shape[:2]
                stem = f"{video_path.stem}_f{frame_index:06d}"
                image_path = image_dir / f"{stem}.jpg"
                label_path = label_dir / f"{stem}.txt"
                if not cv2.imwrite(str(image_path), frame):
                    raise OSError(f"Could not write image: {image_path}")

                labels = []
                if row["Visibility"] == "1":
                    center_x = float(row["X"])
                    center_y = float(row["Y"])
                    half_size = SHUTTLE_BOX_SIZE / 2
                    left = max(0.0, center_x - half_size)
                    top = max(0.0, center_y - half_size)
                    right = min(float(image_width), center_x + half_size)
                    bottom = min(float(image_height), center_y + half_size)
                    box_x = (left + right) / 2 / image_width
                    box_y = (top + bottom) / 2 / image_height
                    box_width = (right - left) / image_width
                    box_height = (bottom - top) / image_height
                    labels.append(
                        f"0 {box_x:.6f} {box_y:.6f} {box_width:.6f} {box_height:.6f}"
                    )

                label_path.write_text("\n".join(labels), encoding="utf-8")
                saved_count += 1

            frame_index += 1

        capture.release()

    return saved_count


video_pairs = sorted(video_csv_pairs, key=lambda pair: pair[0].name)
random.Random(42).shuffle(video_pairs)
validation_count = max(1, round(len(video_pairs) * 0.2))
validation_pairs = video_pairs[:validation_count]
training_pairs = video_pairs[validation_count:]

training_images = write_yolo_split(training_pairs, "train")
validation_images = write_yolo_split(validation_pairs, "val")

data_yaml = dataset_output / "data.yaml"
data_yaml.write_text(
    yaml.safe_dump(
        {
            "path": str(dataset_output),
            "train": "images/train",
            "val": "images/val",
            "names": ["shuttlecock"],
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(f"Training images: {training_images}")
print(f"Validation images: {validation_images}")
print(f"YOLO dataset config: {data_yaml}")

In [ ]:
import torch
from ultralytics import YOLO

shuttle_model = YOLO(str(project_root / "yolov8x.pt"))
shuttle_model.train(
    data=str(data_yaml),
    epochs=30,
    imgsz=960,
    batch=1 if not torch.cuda.is_available() else 8,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=0,
    patience=8,
    close_mosaic=5,
    project=str(project_root / "runs" / "detect"),
    name="shuttlecock",
    exist_ok=True,
)
